# 6.6: Sinusoidal encodings: norm, rotation and offset checks


In [ ]:
import torch

def sinusoidal_pe(n_pos, d):
    pos = torch.arange(n_pos, dtype=torch.float64)[:, None]
    omega = 10000 ** (-torch.arange(0, d, 2, dtype=torch.float64) / d)   # (d/2,)
    pe = torch.zeros(n_pos, d, dtype=torch.float64)
    pe[:, 0::2] = torch.sin(pos * omega)
    pe[:, 1::2] = torch.cos(pos * omega)
    return pe, omega

d = 64
pe, omega = sinusoidal_pe(200, d)
print(torch.allclose(pe.norm(dim=1) ** 2, torch.full((200,), d / 2, dtype=torch.float64)))  # equal norms

# PE_{p+k} = R_k PE_p with a block-diagonal rotation R_k independent of p
k = 7
R = torch.zeros(d, d, dtype=torch.float64)
for i, w in enumerate(omega):
    c, s = torch.cos(w * k), torch.sin(w * k)
    R[2 * i: 2 * i + 2, 2 * i: 2 * i + 2] = torch.tensor([[c, s], [-s, c]])
print(torch.allclose(pe[k:] , pe[:-k] @ R.T))                          # True for every p

# The dot product depends only on the offset
print(torch.allclose(pe[10] @ pe[10 + k], pe[100] @ pe[100 + k]))       # True
